# LightGBM + Meta-Labeling + Ensemble Primary + Joint Thresholds

Sprint 1.5 + Sprint 2 из дорожной карты. Объединение трёх улучшений:

## Sprint 2: Ensemble Primary

Заменяем single-LightGBM Primary на ансамбль **LightGBM + CatBoost +
ExtraTrees**. Каждый алгоритм имеет свой inductive bias; усреднение
снижает variance в ~1.7× (Tukey 1959, Breiman 1996), что критично на
noisy сигнале AUC ~0.6.

Бонус: `std` predictions = disagreement между моделями = **epistemic UQ**.

## Sprint 1.5: Joint (T_prim, T_meta) thresholds

Старый sweep T_meta ∈ {0.5, 0.55, ...} на h=6 убивал ВСЕ trades, потому
что meta-distribution для коротких горизонтов не достигает 0.5
(mean meta для h=6 = 0.157). 

Fix: **percentile-based meta-thresholds** — top-{5,10,15,20,30,40,50}%
от val-meta-распределения. Это **гарантирует непустой n_buy** на любом
горизонте. Plus joint sweep по сетке (T_prim × T_meta_pct), maximizing
val mean(lr - cost).

## Ожидаемый эффект (cumulative)

Baseline (Sprint 1, single LightGBM primary + meta): Sharpe 1.85 на h=48.

- Sprint 1.5 (joint thresholds): h=6/h=12 оживают (n_buy > 0)
- Sprint 2 (ensemble): variance reduction → +5-15% Sharpe на каждом горизонте

Цель: Sharpe ≥ **2.0** на хотя бы одном горизонте, n_buy ≥ 100 (статистически стабильно).

Время: ~25-35 минут на Colab CPU (3 GBM × 4 horizons × OOF + final + meta).

## 0. Окружение

In [ ]:
from __future__ import annotations

import os, subprocess
from pathlib import Path

try:
    import google.colab  # type: ignore  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REPO_OWNER = 'Pozdnyakof'
REPO_NAME = 'lstm_exchange_predictor'
REPO_BRANCH = 'main'

if IN_COLAB:
    PROJECT_ROOT = Path('/content') / REPO_NAME
    env = os.environ.copy(); env['GIT_TERMINAL_PROMPT'] = '0'
    if PROJECT_ROOT.exists():
        subprocess.run(['git', '-C', str(PROJECT_ROOT), 'fetch', 'origin', REPO_BRANCH], check=True, env=env)
        subprocess.run(['git', '-C', str(PROJECT_ROOT), 'reset', '--hard', f'origin/{REPO_BRANCH}'], check=True, env=env)
    else:
        subprocess.run(['git', 'clone', '--depth', '1', '-b', REPO_BRANCH,
                        f'https://github.com/{REPO_OWNER}/{REPO_NAME}.git', str(PROJECT_ROOT)],
                       check=True, env=env)
else:
    PROJECT_ROOT = Path.cwd().parent
PROJECT_ROOT = PROJECT_ROOT.resolve()
print(f'PROJECT_ROOT: {PROJECT_ROOT}')

In [ ]:
if IN_COLAB:
    subprocess.run(['pip', 'install', '--quiet',
                    'pandas>=2.1', 'pyarrow>=15.0', 'scikit-learn>=1.4',
                    'lightgbm>=4.0', 'catboost>=1.2', 'tqdm>=4.66',
                    'matplotlib>=3.7'],
                   check=True)
    print('Deps OK (with CatBoost)')

In [ ]:
import sys
SRC = PROJECT_ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s: %(message)s')

import dataclasses
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score

from graduate_work.config import default_config

TICKERS = (
    'SBER', 'VTBR', 'GAZP', 'LKOH', 'GMKN', 'ROSN', 'NVTK',
    'MGNT', 'TATN', 'MTSS', 'MOEX', 'NLMK', 'CHMF', 'ALRS',
)

cfg = default_config()
cfg = dataclasses.replace(cfg, data=dataclasses.replace(cfg.data, tickers=TICKERS))
print(f'tickers={cfg.data.tickers}\nhorizons={cfg.data.horizons}')

## 1. Drive + загрузка + features (тот же pipeline что в lightgbm_meta_labeling)

In [ ]:
import shutil
DRIVE_BASE = Path('/content/drive/MyDrive/lstm_exchange')
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    src_dir = DRIVE_BASE / 'data' / 'raw' / 'Algopack'
    if not src_dir.exists():
        src_dir = DRIVE_BASE / 'data' / 'raw'
    if src_dir.exists():
        shutil.copytree(src_dir, cfg.paths.data_raw, dirs_exist_ok=True)
        print(f'Скопировано из {src_dir}')
for sub in ['algopack/tradestats', 'algopack/orderstats', 'algopack/obstats', 'algopack/hi2']:
    p = cfg.paths.data_raw / sub
    if p.exists():
        n = sum(1 for _ in p.glob('*.csv'))
        print(f'  {sub}: {n} файлов')

In [ ]:
from graduate_work.features.algopack_features import (
    obstats_features, orderstats_features, tradestats_features,
    order_to_trade_ratio, hi2_features,
)
from graduate_work.features.targets import cost_aware_classification_labels, lr_columns
from graduate_work.features.cross_sectional import add_cross_sectional_features


def _load_csv_indexed(path: Path) -> pd.DataFrame:
    if not path.exists():
        return pd.DataFrame()
    df = pd.read_csv(path)
    if df.empty or 'tradedate' not in df.columns:
        return df
    if 'tradetime' in df.columns:
        ts = pd.to_datetime(df['tradedate'].astype(str) + ' ' + df['tradetime'].astype(str),
                            utc=True, errors='coerce')
    else:
        ts = pd.to_datetime(df['tradedate'], utc=True, errors='coerce')
    df.index = ts
    df.index.name = 'begin'
    return df.dropna(how='all').sort_index()


ts_data, os_data, ob_data, hi2_data = {}, {}, {}, {}
for ticker in TICKERS:
    ts_data[ticker] = _load_csv_indexed(cfg.paths.data_raw / 'algopack' / 'tradestats' / f'{ticker}.csv')
    os_data[ticker] = _load_csv_indexed(cfg.paths.data_raw / 'algopack' / 'orderstats' / f'{ticker}.csv')
    ob_data[ticker] = _load_csv_indexed(cfg.paths.data_raw / 'algopack' / 'obstats' / f'{ticker}.csv')
    hi2_data[ticker] = _load_csv_indexed(cfg.paths.data_raw / 'algopack' / 'hi2' / f'{ticker}.csv')

loaded = {t: len(ts_data[t]) for t in TICKERS if not ts_data[t].empty}
missing = set(TICKERS) - set(loaded.keys())
if missing:
    TICKERS = tuple(t for t in TICKERS if t not in missing)
    print(f'MISSING: {sorted(missing)}; using {len(TICKERS)} tickers')


def _ts_to_ohlcv(ts: pd.DataFrame, ticker: str) -> pd.DataFrame:
    out = pd.DataFrame(index=ts.index)
    out['open']   = ts['pr_open'].astype(float)
    out['high']   = ts['pr_high'].astype(float)
    out['low']    = ts['pr_low'].astype(float)
    out['close']  = ts['pr_close'].astype(float)
    out['volume'] = ts['vol'].astype(float)
    out['value']  = ts['val'].astype(float)
    out['vwap']   = ts['pr_vwap'].astype(float)
    for col in ['sec_pr_open', 'sec_pr_high', 'sec_pr_low', 'sec_pr_close']:
        if col in ts.columns:
            out[col] = ts[col].astype(float)
    out['ticker'] = ticker
    return out


def _build_log_returns(close: pd.Series) -> pd.DataFrame:
    lr = np.log(close / close.shift(1))
    return pd.DataFrame({
        'lr_1':  lr,
        'lr_5':  lr.rolling(5).sum(),
        'lr_15': lr.rolling(15).sum(),
        'lr_30': lr.rolling(30).sum(),
    }, index=close.index)


def _build_sec_features(feat: pd.DataFrame) -> pd.DataFrame:
    if 'sec_pr_close' not in feat.columns:
        return pd.DataFrame(index=feat.index)
    out = pd.DataFrame(index=feat.index)
    sec_close = feat['sec_pr_close'].astype(float)
    out['sec_rel_strength'] = (feat['close'] / sec_close.replace(0, np.nan)).fillna(1.0) - 1.0
    sec_lr = np.log(sec_close / sec_close.shift(1))
    out['sec_lr_1'] = sec_lr.fillna(0.0)
    out['sec_lr_5'] = sec_lr.rolling(5).sum().fillna(0.0)
    out['sec_lr_15'] = sec_lr.rolling(15).sum().fillna(0.0)
    own_lr = np.log(feat['close'] / feat['close'].shift(1))
    out['sec_lr_residual'] = (own_lr - sec_lr).fillna(0.0)
    return out


def build_per_ticker(ticker: str) -> pd.DataFrame:
    ts_df = ts_data[ticker]
    if ts_df.empty:
        return pd.DataFrame()
    feat = _ts_to_ohlcv(ts_df, ticker)
    feat = feat.join(_build_log_returns(feat['close']), how='left')
    feat = feat.join(_build_sec_features(feat), how='left')
    feat = feat.join(tradestats_features(ts_df), how='left')
    if not os_data[ticker].empty:
        feat = feat.join(orderstats_features(os_data[ticker]), how='left')
    if not ob_data[ticker].empty:
        feat = feat.join(obstats_features(ob_data[ticker]), how='left')
    if not os_data[ticker].empty:
        feat = feat.join(order_to_trade_ratio(os_data[ticker], ts_data[ticker]), how='left')
    if not hi2_data[ticker].empty:
        feat = feat.join(hi2_features(hi2_data[ticker], feat.index), how='left')
    return feat.fillna(0.0)


per_ticker = {t: build_per_ticker(t) for t in TICKERS}
per_ticker = {t: df for t, df in per_ticker.items() if not df.empty}
XS_FEATURES = [
    'aps_vol_imb', 'aps_disb', 'aps_vwap_premium',
    'aps_imb_vol_bbo', 'aps_pr_change_bp', 'aps_spread_bbo_bp',
]
per_ticker_xs = add_cross_sectional_features(
    per_ticker, base_features=XS_FEATURES,
    rank=True, zscore=True, relative_mode='diff',
)
print(f'Per-ticker frames: {len(per_ticker_xs)}, sample shape: {next(iter(per_ticker_xs.values())).shape}')

In [ ]:
full = pd.concat(per_ticker_xs.values()).sort_index()
out_parts = []
costs = cfg.trading.commission_rate + cfg.trading.slippage_rate
for ticker, sub in full.groupby('ticker'):
    labels = cost_aware_classification_labels(
        open_price=sub['open'], close_price=sub['close'],
        horizons=cfg.data.horizons,
        entry_cost=costs, exit_cost=costs,
        label_smoothing=0.0, direction='long',
    )
    out_parts.append(sub.join(labels, how='left'))
full_with_targets = pd.concat(out_parts).sort_values(['ticker']).sort_index(kind='stable')

target_cols = [f'target_h{h}' for h in cfg.data.horizons]
lr_cols_list = lr_columns(cfg.data.horizons)
feature_cols = [c for c in full_with_targets.columns
                if c not in ('ticker',) and c not in target_cols and c not in lr_cols_list]
print(f'Multi-ticker frame: {full_with_targets.shape}, features: {len(feature_cols)}')

from graduate_work.features.pipeline import chronological_split
train_df, val_df, test_df = chronological_split(
    full_with_targets,
    cfg.data.train_ratio, cfg.data.val_ratio,
    purge_horizon=max(cfg.data.horizons),
)
test_prices_raw = test_df[['open', 'close', 'ticker']].copy()
print(f'train: {len(train_df)}, val: {len(val_df)}, test: {len(test_df)}')

## 2. Pipeline: Ensemble Primary + Meta-Labeling

Архитектура: `MetaLabelingPipeline` принимает любой Primary-объект,
имеющий интерфейс fit/predict (`LightGBMPipeline` или `EnsemblePipeline`).

Здесь Primary = **EnsemblePipeline** (3 GBM), Meta = LightGBM как раньше.

Только нужен ручной orchestration, потому что MetaLabelingPipeline сейчас
под single-LightGBM. Сделаем явные шаги.

In [ ]:
from graduate_work.model.ensemble_pipeline import (
    BaseModelConfig, EnsemblePipeline,
    default_lightgbm_config, default_catboost_config, default_extratrees_config,
)
from graduate_work.model.lgbm_pipeline import LightGBMConfig, LightGBMPipeline
from graduate_work.model.meta_labeling import (
    add_primary_predictions_wide, build_meta_targets,
    joint_max_pnl_thresholds,
)

# === Sprint 2: Ensemble Primary ===
primary = EnsemblePipeline(
    horizons=cfg.data.horizons,
    feature_cols=feature_cols,
    base_configs=[
        default_lightgbm_config(),
        default_catboost_config(),
        default_extratrees_config(),
    ],
)
primary.fit(train_df, val_df)

print('\n=== Ensemble Primary results ===')
for h, r in primary.models.items():
    auc_str = f'{r.val_auc:.4f}' if r.val_auc is not None else 'n/a'
    print(f'  h={h}: val_log_loss={r.val_log_loss:.4f}, val_auc={auc_str}, '
          f'base_models={r.base_types}')

In [ ]:
# === Sprint 1 (meta-labeling): OOF на train с Ensemble Primary ===
# Делаем k-fold CV ensemble отдельно (т.к. compute_lgbm_oof работает
# только с LightGBM).
from sklearn.model_selection import TimeSeriesSplit
import copy

n_oof = 5
n_train = len(train_df)
oof_arrays = {f'primary_h{h}': np.full(n_train, np.nan, dtype=np.float32)
              for h in cfg.data.horizons}

tscv = TimeSeriesSplit(n_splits=n_oof)
for fold_idx, (tr_idx, te_idx) in enumerate(tscv.split(np.arange(n_train))):
    print(f'OOF fold {fold_idx + 1}/{n_oof}')
    tr_sub = train_df.iloc[tr_idx]
    te_sub = train_df.iloc[te_idx]
    fold_pipeline = EnsemblePipeline(
        horizons=cfg.data.horizons,
        feature_cols=feature_cols,
        base_configs=copy.deepcopy(primary.base_configs),
    )
    fold_pipeline.fit(tr_sub, te_sub)
    fold_preds = fold_pipeline.predict(te_sub)
    fold_preds['timestamp'] = pd.to_datetime(fold_preds['timestamp'], utc=True)
    # Map back to train_df positions through (timestamp, ticker, horizon)
    te_sub_meta = pd.DataFrame({
        'timestamp': pd.to_datetime(te_sub.index, utc=True),
        'ticker': te_sub['ticker'].values,
        'pos_in_train': te_idx,
    })
    for h in cfg.data.horizons:
        h_block = fold_preds[fold_preds['horizon'] == h]
        merged = h_block.merge(te_sub_meta, on=['timestamp', 'ticker'], how='inner')
        oof_arrays[f'primary_h{h}'][merged['pos_in_train'].values] = (
            merged['mean'].values.astype(np.float32)
        )

oof = pd.DataFrame(oof_arrays, index=train_df.index)
for h in cfg.data.horizons:
    n_valid = oof[f'primary_h{h}'].notna().sum()
    print(f'  primary_h{h}: {n_valid} valid OOF predictions out of {n_train}')

In [ ]:
# === Build meta train + val ===
META_CONTEXT_FEATURES = [
    'aps_intra_vol_bp', 'aps_spread_bbo_bp', 'aps_spread_lv10_bp',
    'lr_5', 'lr_15', 'lr_30',
    'aps_levels_total', 'aps_imb_vol_bbo', 'aps_levels_imb',
    'aps_hi2_hhi_volume', 'aps_hi2_hhi_aggressive', 'aps_hi2_hhi_passive',
    'sec_rel_strength', 'sec_lr_residual',
    'aps_imb_vol_bbo_xrank', 'aps_disb_xrank',
]
META_CONTEXT_FEATURES = [c for c in META_CONTEXT_FEATURES if c in feature_cols]
META_FEATURES = META_CONTEXT_FEATURES + [f'primary_h{h}' for h in cfg.data.horizons]

# Meta train: train_df + OOF primary + meta targets
meta_train = train_df.copy()
for h in cfg.data.horizons:
    meta_train[f'primary_h{h}'] = oof[f'primary_h{h}']
meta_train = build_meta_targets(
    meta_train, cfg.data.horizons,
    cost_per_trade=costs, profit_multiplier=2.0,
)
# Replace target_h{h} → meta_target_h{h} for LightGBMPipeline
meta_train_clean = meta_train.copy()
for h in cfg.data.horizons:
    meta_train_clean[f'target_h{h}'] = meta_train[f'meta_target_h{h}']
# Drop NaN OOF rows (первые 1/(n_oof+1))
oof_mask = meta_train_clean[[f'primary_h{h}' for h in cfg.data.horizons]].notna().all(axis=1)
meta_train_clean = meta_train_clean[oof_mask]
print(f'Meta train rows after OOF NaN drop: {len(meta_train_clean)}')

# Meta val: val_df + final-primary preds
val_primary_preds = primary.predict(val_df)
val_primary_preds['timestamp'] = pd.to_datetime(val_primary_preds['timestamp'], utc=True)
meta_val = add_primary_predictions_wide(
    val_df, val_primary_preds, cfg.data.horizons,
)
meta_val = build_meta_targets(
    meta_val, cfg.data.horizons,
    cost_per_trade=costs, profit_multiplier=2.0,
)
for h in cfg.data.horizons:
    meta_val[f'target_h{h}'] = meta_val[f'meta_target_h{h}']

# Train Meta (single LightGBM, как в Sprint 1 — фокус на регуляризации)
meta_cfg = LightGBMConfig(
    n_estimators=300, num_leaves=15, learning_rate=0.03,
    min_child_samples=500, reg_lambda=2.0, reg_alpha=0.5,
    early_stopping_rounds=30, random_state=42,
)
meta = LightGBMPipeline(
    horizons=cfg.data.horizons,
    feature_cols=META_FEATURES,
    cfg=meta_cfg,
)
meta.fit(meta_train_clean, meta_val)

print('\n=== Meta results (on top of Ensemble Primary) ===')
for h, r in meta.models.items():
    auc_str = f'{r.val_auc:.4f}' if r.val_auc is not None else 'n/a'
    print(f'  h={h}: val_log_loss={r.val_log_loss:.4f}, val_auc={auc_str}')

# Сохраняем
ckpt_dir = cfg.paths.checkpoints / 'meta_ensemble'
ckpt_dir.mkdir(parents=True, exist_ok=True)
meta.save(ckpt_dir / 'meta')
print(f'Meta saved to {ckpt_dir / "meta"}')

## 3. Predict on test + diagnostic

In [ ]:
# Test predictions: ensemble primary → meta
test_primary_preds = primary.predict(test_df)
test_primary_preds['timestamp'] = pd.to_datetime(test_primary_preds['timestamp'], utc=True)
test_with_primary = add_primary_predictions_wide(
    test_df, test_primary_preds, cfg.data.horizons,
)
test_meta_preds = meta.predict(test_with_primary)
test_meta_preds['timestamp'] = pd.to_datetime(test_meta_preds['timestamp'], utc=True)

test_long_index = pd.to_datetime(test_df.index, utc=True)

print('=== Per-horizon: Primary AUC vs Meta AUC on test ===')
print(f'{"horizon":>8} | {"P(UP)":>7} | {"prim mean":>9} | {"prim std":>9} | {"meta mean":>9} | {"prim AUC":>9} | {"meta AUC":>9}')
for h in cfg.data.horizons:
    target_col = f'target_h{h}'
    lr_col = f'lr_h{h}'
    if target_col not in test_df.columns:
        continue
    mask = test_df[target_col].notna()
    if not mask.any():
        continue
    sub = pd.DataFrame({
        'timestamp': test_long_index[mask.to_numpy()],
        'ticker': test_df.loc[mask, 'ticker'].to_numpy(),
        'target': test_df.loc[mask, target_col].astype(int).to_numpy(),
        'lr': test_df.loc[mask, lr_col].to_numpy(),
    })
    h_prim = test_primary_preds[test_primary_preds['horizon'] == h]
    h_meta = test_meta_preds[test_meta_preds['horizon'] == h]
    merged_p = h_prim.merge(sub, on=['timestamp', 'ticker'], how='inner')
    merged_m = h_meta.merge(sub, on=['timestamp', 'ticker'], how='inner')
    p_up = float(merged_p['target'].mean())
    auc_prim = roc_auc_score(merged_p['target'], merged_p['mean']) if merged_p['target'].nunique() > 1 else float('nan')
    meta_target = (merged_m['lr'] > 2 * costs).astype(int)
    auc_meta = roc_auc_score(meta_target, merged_m['mean']) if meta_target.nunique() > 1 else float('nan')
    prim_std = float(merged_p['std'].mean())  # NEW: std из ensemble disagreement
    print(f'{h:>8} | {p_up:>7.3f} | {merged_p["mean"].mean():>9.4f} | {prim_std:>9.4f} | '
          f'{merged_m["mean"].mean():>9.4f} | {auc_prim:>9.4f} | {auc_meta:>9.4f}')

print('\nprim std = ensemble disagreement (новая фича Sprint 2). Высокий std → нужна осторожность.')

## 4. Joint thresholds optimization (Sprint 1.5)

Sweep по сетке (T_prim × T_meta_percentile), выбираем максимизирующее val PnL.

In [ ]:
# Build long-form val predictions for joint optimization
val_primary_preds_full = primary.predict(val_df)
val_primary_preds_full['timestamp'] = pd.to_datetime(val_primary_preds_full['timestamp'], utc=True)
val_with_primary = add_primary_predictions_wide(
    val_df, val_primary_preds_full, cfg.data.horizons,
)
val_meta_preds = meta.predict(val_with_primary)
val_meta_preds['timestamp'] = pd.to_datetime(val_meta_preds['timestamp'], utc=True)

# Long-form val_lr_targets
val_lr_rows = []
for h in cfg.data.horizons:
    lr_col = f'lr_h{h}'
    sub = val_df[val_df[f'target_h{h}'].notna() & val_df[lr_col].notna()]
    for ts, row in sub.iterrows():
        val_lr_rows.append({
            'timestamp': pd.to_datetime(ts, utc=True), 'ticker': row['ticker'],
            'horizon': int(h), 'actual': float(row[lr_col]),
        })
val_lr_targets = pd.DataFrame(val_lr_rows)
val_lr_targets['timestamp'] = pd.to_datetime(val_lr_targets['timestamp'], utc=True)

# Joint optimization per horizon
optimal_thresholds = {}
for h in cfg.data.horizons:
    T_prim, T_meta_abs, sweep = joint_max_pnl_thresholds(
        val_primary_preds_full, val_meta_preds, val_lr_targets,
        horizon=h,
        primary_thresholds=(0.45, 0.50, 0.55, 0.60),
        meta_percentiles=(5.0, 10.0, 15.0, 20.0, 30.0, 40.0, 50.0),
        cost_per_trade=2 * costs,
        min_trades=30,
    )
    optimal_thresholds[h] = (T_prim, T_meta_abs)
    print(f'h={h}: T_prim={T_prim:.3f}, T_meta_abs={T_meta_abs:.4f}')
    # Top-3 от sweep table для inspection
    sweep_df = pd.DataFrame(sweep)
    sweep_df = sweep_df.dropna(subset=['mean_pnl']).sort_values('mean_pnl', ascending=False)
    if not sweep_df.empty:
        print(f'  Top-3 sweep: ')
        for _, row in sweep_df.head(3).iterrows():
            print(f'    T_prim={row["T_prim"]:.2f}, meta_pct={row["meta_pct"]:.0f}%, '
                  f'n_trades={int(row["n_trades"])}, pnl={row["mean_pnl"]:.5f}')

## 5. Backtest: ensemble × meta × joint thresholds

In [ ]:
from graduate_work.backtest import compute_metrics, run_backtest
from graduate_work.strategy import signals_per_horizon_threshold

bars_per_year = cfg.data.bars_per_year
cost_per_trade = 2.0 * costs


def _bt(signals: pd.DataFrame) -> dict:
    n_buy = int((signals['action'] == 'BUY').sum())
    if n_buy == 0:
        return {'n_buy': 0, 'sharpe': 0.0, 'total_return': 0.0,
                'win_rate': 0.0, 'max_dd': 0.0}
    bt = run_backtest(signals, test_prices_raw, cfg.trading)
    m = compute_metrics(bt.equity, bt.trades, periods_per_year=bars_per_year)
    return {'n_buy': n_buy, 'sharpe': float(m['sharpe']),
            'total_return': float(m['total_return']),
            'win_rate': float(m['win_rate']),
            'max_dd': float(m['max_drawdown'])}


rows = []
for h in cfg.data.horizons:
    if h not in primary.models or h not in meta.models:
        continue
    T_prim, T_meta_abs = optimal_thresholds[h]
    if not np.isfinite(T_meta_abs):
        # Без meta — fallback к primary-only
        sig = signals_per_horizon_threshold(test_primary_preds, h, threshold=T_prim)
        rows.append({
            'horizon': h, 'strategy': 'primary_only_fallback',
            'T_prim': T_prim, 'T_meta_abs': T_meta_abs, **_bt(sig),
        })
        continue
    # Combined: primary > T_prim AND meta > T_meta_abs
    merged_pm = test_primary_preds[test_primary_preds['horizon'] == h].merge(
        test_meta_preds[test_meta_preds['horizon'] == h],
        on=['timestamp', 'ticker', 'horizon'],
        suffixes=('_prim', '_meta'),
    )
    ok = (merged_pm['mean_prim'] > T_prim) & (merged_pm['mean_meta'] > T_meta_abs)
    merged_filt = merged_pm[ok].copy()
    merged_filt['mean'] = merged_filt['mean_prim']
    merged_filt['std'] = merged_filt.get('std_prim', 0.0)
    sig = signals_per_horizon_threshold(merged_filt, h, threshold=T_prim - 0.01)
    rows.append({
        'horizon': h, 'strategy': 'ensemble_meta_joint',
        'T_prim': T_prim, 'T_meta_abs': T_meta_abs, **_bt(sig),
    })
    # Также добавим baseline: primary-only с T_prim (для сравнения)
    sig_base = signals_per_horizon_threshold(test_primary_preds, h, threshold=T_prim)
    rows.append({
        'horizon': h, 'strategy': 'ensemble_primary_only',
        'T_prim': T_prim, 'T_meta_abs': None, **_bt(sig_base),
    })

results = pd.DataFrame(rows)
print('\n=== Test backtest results ===')
print(results.to_string(index=False, float_format=lambda x: f'{x:.4f}' if isinstance(x, float) else str(x)))

## 6. Verdict + сравнение с предыдущими baselines

In [ ]:
positive = results[results['sharpe'] > 0].sort_values('sharpe', ascending=False)
if not positive.empty:
    best = positive.iloc[0]
    print(f'\nBEST: {best["strategy"]} @ h={best["horizon"]}: '
          f'Sharpe={best["sharpe"]:.3f}, return={best["total_return"]*100:.2f}%, '
          f'n_BUY={int(best["n_buy"])}, win_rate={best["win_rate"]:.3f}')
    print('\nСравнение vs предыдущие baselines:')
    print('  Sprint 3 (single LightGBM, 2 tickers):  Sharpe=0.98, n=439')
    print('  Sprint 1 (LightGBM Primary + Meta):     Sharpe=1.85, n=120 (best h=48)')
    print(f'  Now (Ensemble + Meta + Joint):           Sharpe={best["sharpe"]:.2f}, '
          f'n={int(best["n_buy"])} (best h={int(best["horizon"])})')
    if best['sharpe'] > 1.85:
        improvement = (best['sharpe'] - 1.85) / 1.85 * 100
        print(f'  → Sprint 2 improvement vs Sprint 1: +{improvement:.1f}%')
else:
    least_bad = results.sort_values('sharpe', ascending=False).iloc[0]
    print(f'WARNING: best Sharpe = {least_bad["sharpe"]:.3f} (negative или = 0).')
    print('Проверь meta AUC на test и thresholds.')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

# Sharpe per (strategy × horizon)
ax = axes[0]
pivot_sharpe = results.pivot_table(index='strategy', columns='horizon', values='sharpe')
pivot_sharpe.plot.bar(ax=ax)
ax.axhline(0, color='black', lw=0.8)
ax.axhline(1.85, color='green', ls='--', lw=0.8, label='Sprint 1 best (1.85)')
ax.axhline(0.98, color='gray', ls=':', lw=0.8, label='Sprint 3 baseline (0.98)')
ax.set_title('Sharpe per (strategy × horizon)')
ax.legend()
ax.grid(alpha=0.3)

# Distribution: ensemble disagreement (std) — диагностика
ax = axes[1]
for h in cfg.data.horizons:
    h_block = test_primary_preds[test_primary_preds['horizon'] == h]
    ax.hist(h_block['std'], bins=40, alpha=0.5, label=f'h={h}')
ax.set_title('Ensemble disagreement (std) on test')
ax.set_xlabel('std (между LGB/CB/ET)')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print('\nЕсли std маленький (<0.02) → модели согласны → можно агрессивнее торговать.')
print('Если std большой → есть disagreement → стоит фильтровать «низкоуверенные» сигналы.')